# CADI Deep Learning - Semana 11
# # Semana 11:  Implementación de una Red Neuronal Recurrente (RNN) para la Predicción de Series de Tiempo en Google Colab 
# **Estudiante:** Jenifer Cárdenas Aguilera

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# 1. Generamos una serie de tiempo: una tendencia que sube con una onda (estacionalidad)
time = np.arange(0, 400, 0.5)
data = 0.2 * time + 10 * np.sin(time) + np.random.randn(len(time))

# Normalización: Es vital en RNNs para que los valores estén entre 0 y 1
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data.reshape(-1, 1))

plt.figure(figsize=(10, 4))
plt.plot(data)
plt.title("Serie de Tiempo: Ventas Simuladas")
plt.show()

In [ ]:
# 2. Función para crear "ventanas" de tiempo (usar el pasado para predecir el futuro)
def crear_ventanas(dataset, ventana=10):
    X, y = [], []
    for i in range(len(dataset) - ventana):
        X.append(dataset[i:(i + ventana), 0])
        y.append(dataset[i + ventana, 0])
    return np.array(X), np.array(y)

ventana_tiempo = 10
X, y = crear_ventanas(data_scaled, ventana_tiempo)

# Reshape para la RNN: [Muestras, Pasos de tiempo, Características]
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# División: 80% entrenamiento, 20% prueba
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Forma de los datos de entrenamiento: {X_train.shape}")

In [ ]:
# 3. Construcción del modelo recurrente
model = Sequential([
    SimpleRNN(32, input_shape=(ventana_tiempo, 1), activation='relu'),
    Dense(1) # Un solo valor de salida (la predicción)
])

model.compile(optimizer='adam', loss='mean_squared_error')

print("Entrenando la red recurrente...")
history = model.fit(X_train, y_train, epochs=20, batch_size=16, validation_data=(X_test, y_test), verbose=1)

In [ ]:
# 4. Predicción y comparación
predicciones = model.predict(X_test)

# Invertimos la normalización para ver los valores reales
pred_final = scaler.inverse_transform(predicciones)
y_test_final = scaler.inverse_transform(y_test.reshape(-1, 1))

plt.figure(figsize=(10, 5))
plt.plot(y_test_final, label="Valor Real")
plt.plot(pred_final, label="Predicción RNN", linestyle="--")
plt.legend()
plt.title("Comparación: Real vs Predicción")
plt.show()

### Análisis del Desempeño del Modelo

Tras el entrenamiento y la evaluación del modelo con el conjunto de prueba, se pueden observar los siguientes puntos:

* **Captura de Patrones**: El modelo logra seguir la trayectoria de la serie de tiempo, identificando tanto la dirección de la tendencia como los ciclos de oscilación.
* **Generalización**: Al comparar la pérdida (loss) de entrenamiento con la de validación, se observa un comportamiento estable, lo que sugiere que el modelo no presenta un sobreajuste (overfitting) significativo.
* **Precisión**: La visualización final muestra que las predicciones de la RNN se alinean estrechamente con los valores reales, validando la capacidad del modelo para aprender de datos secuenciales.

### Conclusiones Técnicas

1. **Memoria Temporal**: Las Redes Neuronales Recurrentes son herramientas potentes para datos secuenciales debido a su capacidad de mantener un estado de "memoria" sobre eventos pasados para predecir el futuro.

2. **Importancia del Preprocesamiento**: La creación de ventanas temporales (look-back) es el paso más crítico en el pipeline; sin una estructura adecuada de pasado vs. presente, el modelo no puede capturar las dependencias.

3. **Utilidad Práctica**: Este tipo de arquitecturas son esenciales en la industria para resolver problemas de predicción financiera, demanda de inventarios y análisis de sensores.